![](img/logo_ucm.jpg)

# Métricas de Evaluación para Problemas de ML Clásico

Este notebook documenta algunas de las métricas más comunes para evaluar modelos de ML estructuradas según la tipología del problema. 

## Alineamiento entre Métricas Técnicas y Objetivos de Negocio

Es fundamental que exista un **alineamiento entre el equipo de ciencia de datos y los responsables de negocio**, ya que estos últimos serán quienes valoren la utilidad real del sistema y, en muchos casos, lo integren en procesos operativos o decisiones estratégicas.

El equipo de ML debe asumir una **responsabilidad pedagógica**: explicar con claridad en qué consiste cada métrica, qué implicaciones tiene su elección y cómo se interpreta su resultado. Muchas métricas técnicas —como AUC, F1-score o log-loss— pueden no ser intuitivas desde una perspectiva de negocio, por lo que es importante traducirlas a impacto operativo o financiero cuando sea posible.

Además, en muchos casos será necesario **adaptar o incluso reimplementar métricas personalizadas** que reflejen criterios de negocio específicos (por ejemplo, la priorización de ciertos errores, la penalización de falsos negativos en contextos críticos, o la optimización de KPIs no estándar como tasa de conversión, retención o satisfacción de usuario).

Este proceso de colaboración y ajuste garantiza que el modelo no solo sea técnicamente sólido, sino también **interpretable y accionable** para los tomadores de decisiones.


`````{admonition} Comparativas justas
:class: info
Adicionalmente, conviene tener en cuenta que **la mayoría de los procesos donde se va a incorporar un modelo ya cuentan con un sistema preexistente**, muchas veces semi-automático o basado en reglas, que se desea deprecar. Por tanto, es crucial ser **inteligentes en el diseño del problema y en la fase de evaluación** para asegurar que las comparaciones entre el nuevo modelo y el sistema antiguo sean **justas y representativas**. Esto puede implicar reproducir las condiciones originales de producción, simular decisiones operativas o comparar resultados a lo largo de un mismo periodo histórico.
`````



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    log_loss, roc_auc_score, mean_squared_error, mean_absolute_error,
    mean_squared_log_error, r2_score, multilabel_confusion_matrix,
    ndcg_score, average_precision_score, jaccard_score
)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Clasificación Binaria


### Accuracy

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

Proporción de predicciones correctas. Útil cuando las clases están balanceadas.


In [ ]:
y_true = [0, 1, 1, 1, 0]; y_pred = [0, 1, 0, 1, 0]
accuracy_score(y_true, y_pred)

**¿Por qué el accuracy falla con clases desbalanceadas?**

1. Falsa sensación de buen rendimiento: supón un dataset donde el 95% de los casos son negativos (clase 0) y solo el 5% son positivos (clase 1).
Un modelo que siempre predice 0 tendría un accuracy del 95%, pero no estaría detectando ni un solo caso positivo, que probablemente sea el más importante (e.g. fraude, enfermedad, fallo industrial).

2. Ignora la distribución de errores no diferenciando entre los tipos de error: 

- No penaliza más un falso negativo que un falso positivo, cuando en muchos casos uno de los dos puede ser más costoso o crítico.

- No proporciona información sobre la precisión (de los positivos predichos, cuántos son correctos) ni sobre la cobertura o recall (de los positivos reales, cuántos detecta el modelo).

3. No guía bien el aprendizaje del modelo: si el modelo optimiza accuracy en un entorno desbalanceado, aprenderá a favorecer la clase mayoritaria (la más frecuente), ignorando el verdadero objetivo del sistema: identificar correctamente los casos minoritarios y críticos.


**¿Existe algún escenario donde un Falso Positivo sea más costoso que un Falso Negativo?**

Cuando reflexionamos sobre el accuracy siempre pensamos en escenarios en los que un Falso Negativo es mucho más costoso como error que un Falso Positivo, por ejemplo, sistemas muy sensibles, no predeciendo una enfermedad grave versus predecirla y que en una segunda prueba no invasiva se detecte que no es así. Sin embargo, existen casos donde **donde un falso positivo es más costoso que un falso negativo.** Veamos algunos ejemplos: 


| Contexto                      | Falso Positivo (más costoso)                                 | Falso Negativo (menos grave)                                  |
|------------------------------|----------------------------------------------------------------|----------------------------------------------------------------|
| **Finanzas – Detección de fraude** | Bloquear una transacción legítima del cliente                | Dejar pasar una transacción sospechosa de bajo importe        |
| **Justicia predictiva**      | Predecir que alguien reincidirá y denegarle libertad          | Liberar a alguien que puede reincidir pero no lo hace         |
| **Alarmas industriales**     | Parar la línea de producción sin motivo (costes altos)        | Dejar pasar una anomalía no crítica que se corrige sola       |


En general, un sistema excesivamente sensible puede provocar externalidades negativas como: 

- Impacto en la confianza del usuario o cliente: si el sistema da muchas alertas falsas, se vuelve molesto o genera pérdida de reputación.

- Coste operativo innecesario: activar protocolos, investigaciones o paradas por eventos falsos puede costar dinero y tiempo.


### Precision

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

Fracción de positivos predichos que son realmente positivos. Sistemas donde es crucial que las predicciones positivas sean realmente correctas, incluso si se pierden algunos casos positivos (es decir, se prefiere tener menos pero más confiables) como en el caso de **Detección de fraude financiero**.


In [ ]:
precision_score(y_true, y_pred)


### Recall

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

Fracción de positivos reales que fueron predichos correctamente. Sistemas  donde es crítico no dejar escapar casos positivos, incluso si eso implica cometer algunos falsos positivos como en el caso de **Sistemas sanitarios de detección de patologías.**


In [ ]:
recall_score(y_true, y_pred)


### F1 Score

$$
F1 = \frac{2TP}{2TP + FP + FN}
$$

Media armónica de precisión y recall que castiga los desequilibrios entre ambas, y solo da una puntuación alta si ambas son altos. Sistemas que requieren un equilibrio entre ambas métricas sin priorizar la una sobre la otra. Si usamos la media aritmética, podríamos obtener un valor medio engañosamente alto aunque uno de los dos valores sea bajo.


In [ ]:
f1_score(y_true, y_pred)


### Log Loss

$$
\text{LogLoss} = -\left( y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right)
$$

Penaliza predicciones con probabilidades alejadas del valor real. Se utiliza para evaluar la calidad de las probabilidades que un modelo asigna a las clases. No basta con que el sistema acierte la clase: **también importa con cuánta certeza lo hace**. Por tanto, la log loss mide la calibración de las probabilidades.


In [ ]:
y_probs = [0.1, 0.9, 0.4, 0.8, 0.2]
log_loss(y_true, y_probs)


### AUC-ROC

Área bajo la curva ROC. Evalúa discriminación entre clases. mide la capacidad de un modelo de clasificación binaria para distinguir entre clases positivas y negativas. A mayor AUC, mejor capacidad de discriminación.

In [ ]:
roc_auc_score(y_true, y_probs)

### Ejemplo de AUC en Predicción de Impago

Supongamos que un modelo devuelve probabilidades de que un cliente impague (clase 1).  
El objetivo es que **los clientes con mayor probabilidad real de impago estén más arriba en el ranking**.

| Cliente | Probabilidad de impago | Clase real (Impago = 1) |
|---------|-------------------------|--------------------------|
| A       | 0.95                    | 1                        |
| B       | 0.85                    | 0                        |
| C       | 0.80                    | 1                        |
| D       | 0.70                    | 1                        |
| E       | 0.30                    | 0                        |
| F       | 0.10                    | 0                        |



#### ¿Qué hace la AUC?

- Construye una curva **ROC (Receiver Operating Characteristic)**: TPR (*True Positive Rate*) vs. FPR (*False Positive Rate*) a diferentes umbrales.
- La **AUC representa el área bajo esa curva**.
- También puede interpretarse como la **probabilidad de que el modelo ordene correctamente un positivo por encima de un negativo**.

 
En el ejemplo: 

- Comparando todas las parejas posibles (positivos vs. negativos), la AUC será **alta si los positivos tienen mayores probabilidades que los negativos**.
- Si el modelo predice 0.95, 0.80 y 0.70 para los casos positivos, y 0.85, 0.30, 0.10 para los negativos:
  - El modelo comete un fallo con B (0.85, clase 0, más alto que C y D),  
  - Pero acierta en la **mayoría de los pares positivos-negativos**.


#### Interpretación:

- **AUC = 1.0**: clasificación perfecta (todas las clases 1 por encima de las clases 0).
- **AUC = 0.5**: modelo sin poder discriminativo (equivalente a adivinar al azar), es decir, no ordena mejor que una predicción aleatoria.
- **AUC = 0.0**: completamente erróneo (ranking invertido: negativos arriba, positivos abajo).
- En este ejemplo, el **AUC sería alto (~0.92)**, ya que el modelo **sabe distinguir en general entre clientes que impagan y los que no**.



### Matriz de Confusión

Muestra aciertos y errores por clase.



In [ ]:
disp = ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
plt.show()

**Consideraciones clave**

- La métrica debe alinearse con el objetivo del negocio: no siempre accuracy es la mejor.

- En datos desbalanceados, accuracy puede ser engañosa. Es mejor usar F1, AUC o Recall según el contexto.

- Evaluar bien incluye también usar validación cruzada adecuada para obtener métricas robustas y evitar overfitting.

## 2. Clasificación Multiclase


### Precision: Macro, Micro y Weighted

#### Definiciones

- \( TP_i \): verdaderos positivos para la clase \( i \)  
- \( FP_i \): falsos positivos para la clase \( i \)  
- \( C \): número total de clases  
- \( N_i \): número de verdaderos elementos de la clase \( i \)


#### **Precision macro**
Promedia la precisión de cada clase, sin tener en cuenta su frecuencia.

$$
\text{Precision}_{macro} = \frac{1}{C} \sum_{i=1}^{C} \frac{TP_i}{TP_i + FP_i}
$$

- **Ponderación**: todas las clases cuentan por igual.
- **Útil cuando** queremos tratar las clases de forma simétrica, incluso si están desbalanceadas.


#### **Precision micro**
Calcula precisión considerando todos los TP y FP de forma global.

$$
\text{Precision}_{micro} = \frac{\sum_{i=1}^{C} TP_i}{\sum_{i=1}^{C} (TP_i + FP_i)}
$$

- **Ponderación**: por número total de instancias.
- **Útil cuando** importa más el rendimiento general que el balance entre clases.


####  **Precision weighted**
Promedia la precisión por clase, **ponderando según el número de muestras reales por clase**.

$$
\text{Precision}_{weighted} = \sum_{i=1}^{C} \left( \frac{N_i}{\sum_{j=1}^{C} N_j} \cdot \frac{TP_i}{TP_i + FP_i} \right)
$$

- **Ponderación**: proporcional al tamaño de cada clase.
- **Útil cuando** queremos reflejar el impacto de cada clase según su frecuencia en los datos.

In [ ]:
y_true = [0, 1, 2, 1, 0]; y_pred = [0, 2, 1, 1, 0]
print(classification_report(y_true, y_pred))

### ¿Cuál es la mejor métrica para clasificación multiclase?

La elección de la métrica depende del tipo de problema, el balance entre clases y el objetivo del sistema. A continuación, se resumen las opciones más comunes:

---

#### Métricas según el escenario

| Escenario                                           | Métrica recomendada                         | Justificación                                                                 |
|----------------------------------------------------|---------------------------------------------|-------------------------------------------------------------------------------|
| Clases equilibradas                                | **Macro F1**                                | Promedia el F1 de cada clase sin ponderar por tamaño. Evalúa equidad.        |
| Clases desbalanceadas (todas importan)             | **Weighted F1**                             | Promedia el F1 ponderando por número de instancias en cada clase.            |
| Predicción agregada sin distinción de clase        | **Micro F1**                                | Evalúa globalmente TP, FP y FN. Bueno si solo interesa el resultado total.   |
| Modelo da probabilidades calibradas                | **Log Loss**, **AUC (One-vs-Rest)**         | Evalúan la calidad de las probabilidades predichas, no solo la clase.        |


Para problemas críticos donde una clase tiene especial importancia, puede ser mejor evaluar **F1 por clase individual**.

## 3. Regresión


### MAE (Mean Absolute Error)

$$
\text{MAE} = \frac{1}{N} \sum |y_i - \hat{y}_i|
$$

**MAE (Mean Absolute Error)** mide el promedio de las diferencias absolutas entre las predicciones del modelo y los valores reales penalizando todos los errores linealmente (es decir, sin cuadrarlos). 

Tiene la misma unidad que la variable objetivo, es decir, un MAE de 5 significa que, en promedio, el modelo se equivoca 5 unidades (euros, días, kWh...) en cada predicción.

Es robusto frente a valores atípicos (outliers), en comparación con MSE o RMSE, por tanto, no se debe usar MAE si se desea penalizar mucho los errores grandes o si los errores tienen impacto acumulativo o cuadrático.

**Casos típicos:** 

- Predicción del precio de vivienda: se quiere saber el error medio en euros, sin sobrepenalizar errores grandes.
- Forecast de demanda de productos: los errores extremos pueden ser reales (picos de venta) y no deben distorsionar la métrica.

In [ ]:
y_true = [3, -0.5, 2, 7]; y_pred = [2.5, 0.0, 2, 8]
mean_absolute_error(y_true, y_pred)


### MSE (Mean Squared Error)

$$
\text{MSE} = \frac{1}{N} \sum (y_i - \hat{y}_i)^2
$$

**MSE (Error Cuadrático Medio)** mide el **promedio de los errores al cuadrado** entre las predicciones del modelo y los valores reales.

- Penaliza **más fuertemente los errores grandes** que los pequeños.
- Tiene la unidad de la variable al cuadrado (por ejemplo, €² o días²). Un MSE de 16 significa que el **error cuadrático medio es 16 unidades²**, pero no es directamente interpretable como “error promedio”.  
- Es **sensible a outliers**, lo que puede ser una ventaja si esos errores son realmente graves.

No se debe usar MSE si tienes **outliers** no representativos, ya que los sobrepenalizará. Tampoco se debe usar si para el caso de uso, cada unidad de error tiene el **mismo coste**, es decir el error crece linealmente en vez de cuadráticamente. 


Es una buena métrica para modelos sensibles a desviaciones graves o para tareas donde el error tiene un **impacto no lineal**.

In [ ]:
mean_squared_error(y_true, y_pred)

### RMSE (Root Mean Squared Error)


$$
\text{RMSE} = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 }
$$

**RMSE (Raíz del Error Cuadrático Medio)** es la raíz cuadrada del MSE:

- Tiene la **misma unidad** que la variable objetivo.
- Penaliza **errores grandes más que pequeños**, como el MSE.
- Es más **interpretativa que el MSE** porque devuelve el error en escala original. Un RMSE de 7 significa que, en promedio, el modelo se desvía unos **7 unidades (ej: euros, días, grados, etc.)** respecto al valor real, **pero penaliza más los errores grandes que el MAE**.

**Casos típicos:** 

- Estimación del tiempo de entrega (ETA): permite saber cuánto nos desviamos en minutos, penalizando errores grandes.
- Calibración de sensores: una desviación importante puede significar fallo técnico.


En general, **RMSE** combina las ventajas del MSE (penalización cuadrática) con la **interpretabilidad de las unidades originales**.


### RMSLE (Root Mean Squared Logarithmic Error)

$$
\text{RMSLE} = \sqrt{\frac{1}{N} \sum \left(\log(1 + y_i) - \log(1 + \hat{y}_i)\right)^2}
$$

**RMSLE** mide el error cuadrático medio entre los logaritmos de los valores predichos y reales: 

- Penaliza menos los errores grandes que el RMSE, **cuando las predicciones y los valores reales son ambos grandes**.
- Se centra en los **errores relativos o proporcionales**, no absolutos.
- Es ideal cuando **los errores pequeños en magnitudes bajas importan más** que los errores grandes en magnitudes altas.


**Casos típicos:** 

- Predicción de precios con gran variabilidad: importa más acertar en proporción (ej: 10% de error en casas baratas vs. caras).
- Predicción de tráfico en webs/apps: modelar fenómenos de crecimiento rápido o caídas drásticas.



In [ ]:
mean_squared_log_error([1, 2, 3], [1.1, 1.9, 3.2])


### R² Score (Coeficiente de Determinación)

$$
R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}
$$


El **R² score** mide qué proporción de la **variabilidad de la variable dependiente** es explicada por el modelo. Es una métrica global de bondad de ajuste en problemas de regresión. Podemos interpretar la métrica como: 


| Valor de \( R^2 \) | Significado                                                |
|--------------------|------------------------------------------------------------|
| **1.0**            | Predicción perfecta (el modelo explica el 100% de la varianza) |
| **0.0**            | El modelo no mejora respecto a predecir la media           |
| **< 0.0**          | El modelo es peor que predecir siempre la media            |
| **0.7, 0.8, ...**  | Proporción explicada: 70%, 80%, etc. del total de variación |


**¿Qué es el % no explicado?** representa los errores residuales del modelo: la parte de los datos que el modelo no ha podido aprender o predecir correctamente. Debido a: 

- Variabilidad que no se puede capturar con las variables disponibles (información faltante).

- Efectos no modelados (no lineales, interacciones).

- Ruido inherente al proceso (aleatoriedad del fenómeno).

- Outliers o errores de medición.

En general, el **R² score** es útil como **medida global de ajuste**, especialmente cuando quieres saber cuánto del comportamiento de la variable objetivo es capturado por el modelo. Aunque no debe usarse en modelos **no lineales complejos**, si el dataset es muy **pequeño o con outliers**, queremos **comparar modelos con distinto número de variables** (usar **R² ajustado** en su lugar). 


In [ ]:
r2_score(y_true, y_pred)

## 4. Clasificación Multilabel

En la **clasificación multilabel**, una instancia puede estar asociada a **más de una etiqueta simultáneamente**. A diferencia de la clasificación multiclase (donde se predice **una única clase entre varias**), aquí cada muestra puede pertenecer a **cero, una o varias clases al mismo tiempo**.

#### 1. Estructura del espacio de salida

- Si hay \( C \) etiquetas posibles, el espacio de salida tiene \( 2^C \) combinaciones posibles (presencia/ausencia de cada etiqueta).
- Esto genera una explosión combinatoria que **complica el aprendizaje y la evaluación**.

#### 2. Correlaciones entre etiquetas
- Las etiquetas **no son independientes** entre sí (por ejemplo, un texto con etiqueta “deporte” puede tener alta probabilidad de tener también “salud”).
- Ignorar estas dependencias puede llevar a peores predicciones.

#### 3. Métricas más complejas
- Las métricas deben evaluar **conjuntos de etiquetas**, no una clase única:
  - Hamming Loss
  - Subset Accuracy (Exact Match)
  - Precision@k y MAP@k
- Algunas métricas son **muy estrictas** (ej: exact match), mientras otras son **más tolerantes**.

#### 4. Desequilibrio de etiquetas
- No solo hay desbalance entre clases, sino también **frecuencias combinadas** de etiquetas.
- Muchas combinaciones posibles de etiquetas **pueden tener muy pocas muestras**, lo que dificulta el aprendizaje.

#### 5. Modelado especializado
- No basta con aplicar modelos convencionales que quedan fuera del alcance del curso. 
- Se necesitan técnicas como:
  - Reducción a clasificación multiclase 
  - Modelos de redes neuronales con múltiples salidas activadas

Dependiendo de la tipología de problema, existen métricas particulares, por ejemplo para otros ámbitos como el Computer Vision o el procesamiento de lenguaje natural (NLP), queda como ejercicio para el alumno, la búsqueda e identificación de las métricas más usadas puesto que con el auge de ambas tipologías de imagen (imagen & texto), es muy probable que en vayan a toparse con problemas similares durante su vida laboral. 